# Data analysis and dataset preparation

# 0.0 Packages


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
from collections import Counter
from sklearn.preprocessing import MultiLabelBinarizer
from IPython.display import display, HTML

# 1.0 Load data

In [ ]:
df = pd.read_csv("tv2_data.csv", sep=';')

print(df.shape)
df.head()

#2.0 Word count in Play description column

In [ ]:
#Calculates the min, max, and average word count of the text descriptions.
#It also shows a graph of the distribution.
word_counts = []
for text in df['Play description']:
    if pd.notna(text):
        words = str(text).split()
        word_counts.append(len(words))

counts_words = pd.Series(word_counts)

print(f"Min: {counts_words.min()} words")
print(f"Max: {counts_words.max()} words")
print(f"Average: {counts_words.mean():.1f} words")
print(f"Median: {counts_words.median()} words")

fig = px.histogram(
    x=word_counts,
    title="Distribution of the word count in 'Play description'",
    labels={"x": "Word count in the description"},
    nbins=50
    )

#code to show histogram of word distribution counts
fig = px.histogram(
    x=word_counts,
    title="Distribution of the word count in 'Play description'",
    labels={"x": "Word count in the description"},
    nbins=50
    )
fig.show(config={
    'toImageButtonOptions': {
        'format': 'png',
        'filename': 'Distribution of the word count',
        'scale': 3
    }
})

Min: 3 words
Max: 106 words
Average: 32.4 words
Median: 30.0 words


# 3.0 Multi-Label Distribution Complexity Metrics


In [ ]:
#Used to calculate complexity metrics for each label category.
labelcols = ["Genre", "Form", "Målgruppe", "Emne", "Stemning"]

metrics_list = []
figs = []

translations = {"Målgruppe": "Target audience", "Emne": "Topic", "Stemning": "Mood"}

for col in labelcols:
    display_col = translations.get(col, col.title())
    # Extract labels, remove nulls, and prepare as lists
    labels_list = []

    # Iterate through each row in the column
    for value in df[col]:
        if pd.notna(value):
            split_labels = str(value).split(',')
            cleaned_labels = []
            for label in split_labels:
                cleaned_labels.append(label.strip())
            labels_list.append(cleaned_labels)

    #Transform into a binary matrix
    mlb = MultiLabelBinarizer()
    y_matrix = mlb.fit_transform(labels_list)

    #Calculate matrix metrics
    num_rows = y_matrix.shape[0]
    num_labels = y_matrix.shape[1]

    # Cardinality, Density, and Sparsity
    cardinality = np.mean(np.sum(y_matrix, axis=1))
    density = np.sum(y_matrix) / (num_rows * num_labels)
    sparsity = (1 - density) * 100

    label_counts = np.sum(y_matrix, axis=0)
    unique_combos = np.unique(y_matrix, axis=0).shape[0]

    # MeanIR
    max_label_count = np.max(label_counts)
    ir_per_label = max_label_count / label_counts
    mean_ir = np.mean(ir_per_label)

    # Save data to the table
    metrics_list.append({
        "Category": display_col,
        "Unique Labels": num_labels,
        "Cardinality": round(cardinality, 2),
        "Density": round(density, 4),
        "Sparsity (%)": round(sparsity, 1),
        "Label sets": unique_combos,
        "MeanIR": round(mean_ir, 2)
        })

    plot_df = pd.DataFrame({
        "Label": mlb.classes_,
        "Count": label_counts
    }).sort_values("Count", ascending=False)

    fig = px.bar(
        plot_df,
        x="Label",
        y="Count",
        title=f"Label Frequencies: {display_col}",
        text="Count"
        )
    fig.update_layout(xaxis={"categoryorder": "total descending"})
    fig.update_traces(textposition="outside")
    figs.append(fig)

# Display combined table of metrics
metrics_df = pd.DataFrame(metrics_list)
display(HTML(metrics_df.to_html(index=False)))

# Show graphs
for fig in figs:
    title = fig.layout.title.text if fig.layout.title.text else "Plot"
    config = {
        'toImageButtonOptions': {
            'format': 'png',
            'filename': title,
            'scale': 3
        }
    }
    fig.show(config=config)


Category,Unique Labels,Cardinality,Density,Sparsity (%),Label sets,MeanIR
Genre,22,1.80,0.0817,91.8,144,81.24
Form,25,1.59,0.0638,93.6,144,125.55
Target audience,6,1.30,0.2174,78.3,20,9.30
Topic,218,5.26,0.0241,97.6,2000,44.52
Mood,39,3.62,0.0929,90.7,1180,13.42


# 4.0 Label Metrics Before & After tail cut

In [ ]:
#Tests different frequency cutoffs for all label categories and calculates complexity metrics
labelcols = ["Genre", "Form", "Målgruppe", "Emne", "Stemning"]
min_samples_to_test = [0, 3, 5, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100, 120, 140, 160, 180, 200]
translations = {"Målgruppe": "Target audience", "Emne": "Topic", "Stemning": "Mood"}

for col in labelcols:
    display_col = translations.get(col, col.title())
    print(f"Category: {display_col}")
    # Get rows and labels
    all_rows_original = []
    all_labels_in_category = []

    for value in df[col]:
        if pd.notna(value):
            labels_in_row = str(value).split(',')
            row_list = []
            for lbl in labels_in_row:
                clean_lbl = lbl.strip()
                row_list.append(clean_lbl)
                all_labels_in_category.append(clean_lbl)
            all_rows_original.append(row_list)

    # Count labels
    label_counts_series = pd.Series(all_labels_in_category).value_counts()
    total_occurrences_original = len(all_labels_in_category)
    total_unique_original = len(label_counts_series)

    results_for_this_category = []

    # Test cutoffs
    for cutoff in min_samples_to_test:
        step_name = (f"{cutoff}")
        # Find good labels
        valid_labels = []
        for label_name, count in label_counts_series.items():
            if count >= cutoff:
                valid_labels.append(label_name)

        labels_kept_count = len(valid_labels)

        # Clean rows
        filtered_rows = []
        for row in all_rows_original:
            kept_in_row = []
            for label in row:
                if label in valid_labels:
                    kept_in_row.append(label)
            if len(kept_in_row) > 0:
                filtered_rows.append(kept_in_row)

        # Calculate metrics
        if len(filtered_rows) > 0:
            mlb = MultiLabelBinarizer()
            y_matrix = mlb.fit_transform(filtered_rows)

            num_rows = y_matrix.shape[0]
            num_labels = y_matrix.shape[1]
            row_sums = np.sum(y_matrix, axis=1)

            cardinality = np.mean(row_sums)

            total_ones = np.sum(y_matrix)
            density = total_ones / (num_rows * num_labels)
            sparsity = (1 - density) * 100

            label_column_sums = np.sum(y_matrix, axis=0)
            unique_combos = np.unique(y_matrix, axis=0).shape[0]

            # MeanIR
            max_count = np.max(label_column_sums)
            ir_per_label = max_count / label_column_sums
            mean_ir = np.mean(ir_per_label)

            # Calculate % kept
            rows_kept_pct = (num_rows / len(all_rows_original)) * 100
            occurrences_kept_pct = (np.sum(label_column_sums) / total_occurrences_original) * 100
            #Save to dict
            metrics_dict = {
                "Cutoff": step_name,
                "Labels kept": str(labels_kept_count) + " / " + str(total_unique_original),
                "Occurrences kept (%)": round(occurrences_kept_pct, 2),
                "Rows kept (%)": round(rows_kept_pct, 2),
                "Cardinality": round(cardinality, 2),
                "Density": round(density, 4),
                "Sparsity (%)": round(sparsity, 1),
                "Label sets": unique_combos,
                "MeanIR": round(mean_ir, 2)
            }

            results_for_this_category.append(metrics_dict)

    # Make table
    df_results = pd.DataFrame(results_for_this_category)

    ordered_columns = [
        "Cutoff",
        "Labels kept",
        "Occurrences kept (%)",
        "Rows kept (%)",
        "Cardinality",
        "Density",
        "Sparsity (%)",
        "Label sets",
        "MeanIR"
    ]
    df_results = df_results[ordered_columns]
    display(HTML(df_results.to_html(index=False)))
    print("\n")


Category: Genre


Cutoff,Labels kept,Occurrences kept (%),Rows kept (%),Cardinality,Density,Sparsity (%),Label sets,MeanIR
0,22 / 22,100.00,100.00,1.80,0.0817,91.8,144,81.24
3,20 / 22,99.95,100.00,1.80,0.0898,91.0,142,16.37
5,20 / 22,99.95,100.00,1.80,0.0898,91.0,142,16.37
10,19 / 22,99.72,99.63,1.80,0.0947,90.5,140,12.96
20,18 / 22,99.41,99.63,1.79,0.0996,90.0,136,10.30
30,16 / 22,98.19,99.63,1.77,0.1107,88.9,121,7.70
40,15 / 22,97.26,99.12,1.76,0.1176,88.2,113,6.86
50,13 / 22,95.09,99.07,1.73,0.1327,86.7,86,5.24
60,12 / 22,93.75,99.07,1.70,0.1417,85.8,77,4.51
70,10 / 22,90.41,99.02,1.64,0.1641,83.6,48,3.15




Category: Form


Cutoff,Labels kept,Occurrences kept (%),Rows kept (%),Cardinality,Density,Sparsity (%),Label sets,MeanIR
0,25 / 25,100.00,100.00,1.59,0.0638,93.6,144,125.55
3,22 / 25,99.88,100.00,1.59,0.0724,92.8,140,39.61
5,22 / 25,99.88,100.00,1.59,0.0724,92.8,140,39.61
10,18 / 25,99.04,99.91,1.58,0.0878,91.2,128,20.51
20,15 / 25,97.87,99.21,1.57,0.1049,89.5,112,9.89
30,15 / 25,97.87,99.21,1.57,0.1049,89.5,112,9.89
40,13 / 25,95.72,98.65,1.55,0.1190,88.1,92,7.64
50,13 / 25,95.72,98.65,1.55,0.1190,88.1,92,7.64
60,11 / 25,92.69,98.37,1.50,0.1366,86.3,69,5.85
70,10 / 25,90.86,98.00,1.48,0.1479,85.2,63,5.00




Category: Target audience


Cutoff,Labels kept,Occurrences kept (%),Rows kept (%),Cardinality,Density,Sparsity (%),Label sets,MeanIR
0,6 / 6,100.00,100.00,1.30,0.2174,78.3,20,9.30
3,6 / 6,100.00,100.00,1.30,0.2174,78.3,20,9.30
5,6 / 6,100.00,100.00,1.30,0.2174,78.3,20,9.30
10,6 / 6,100.00,100.00,1.30,0.2174,78.3,20,9.30
20,6 / 6,100.00,100.00,1.30,0.2174,78.3,20,9.30
30,6 / 6,100.00,100.00,1.30,0.2174,78.3,20,9.30
40,6 / 6,100.00,100.00,1.30,0.2174,78.3,20,9.30
50,6 / 6,100.00,100.00,1.30,0.2174,78.3,20,9.30
60,6 / 6,100.00,100.00,1.30,0.2174,78.3,20,9.30
70,6 / 6,100.00,100.00,1.30,0.2174,78.3,20,9.30




Category: Topic


Cutoff,Labels kept,Occurrences kept (%),Rows kept (%),Cardinality,Density,Sparsity (%),Label sets,MeanIR
0,218 / 218,100.00,100.00,5.26,0.0241,97.6,2000,44.52
3,206 / 218,99.85,100.00,5.25,0.0255,97.4,2000,26.55
5,196 / 218,99.56,99.95,5.24,0.0267,97.3,1998,20.89
10,173 / 218,98.05,99.95,5.16,0.0298,97.0,1993,15.44
20,137 / 218,93.68,99.95,4.93,0.0360,96.4,1971,10.64
30,97 / 218,85.25,99.54,4.51,0.0465,95.4,1894,7.23
40,76 / 218,78.91,99.40,4.18,0.0550,94.5,1813,5.60
50,64 / 218,74.17,99.21,3.93,0.0615,93.9,1738,4.77
60,52 / 218,68.61,98.84,3.65,0.0703,93.0,1608,3.91
70,46 / 218,65.21,98.61,3.48,0.0757,92.4,1503,3.51




Category: Mood


Cutoff,Labels kept,Occurrences kept (%),Rows kept (%),Cardinality,Density,Sparsity (%),Label sets,MeanIR
0,39 / 39,100.00,100.00,3.62,0.0929,90.7,1180,13.42
3,39 / 39,100.00,100.00,3.62,0.0929,90.7,1180,13.42
5,39 / 39,100.00,100.00,3.62,0.0929,90.7,1180,13.42
10,38 / 39,99.88,100.00,3.62,0.0953,90.5,1178,11.72
20,33 / 39,98.96,100.00,3.59,0.1087,89.1,1155,5.81
30,32 / 39,98.59,100.00,3.57,0.1116,88.8,1143,5.24
40,31 / 39,98.14,100.00,3.56,0.1147,88.5,1127,4.76
50,30 / 39,97.53,100.00,3.53,0.1178,88.2,1115,4.43
60,29 / 39,96.78,100.00,3.51,0.1209,87.9,1103,4.17
70,27 / 39,95.09,99.86,3.45,0.1278,87.2,1047,3.69


#5.0 Compare final cutoffs

In [ ]:
# Compare the original data to the chosen custom cutoffs
custom_cutoffs = {
    "Genre": 70,
    "Form": 70,
    "Målgruppe": 0,
    "Emne": 30,
    "Stemning": 80
}

translations = {"Målgruppe": "Target audience", "Emne": "Topic", "Stemning": "Mood"}

for col, cutoff in custom_cutoffs.items():
    display_col = translations.get(col, col.title())
    print(f"Category: {display_col}")

    #Get all original rows and labels
    all_rows_original = []
    all_labels_in_category = []

    for value in df[col]:
        if pd.notna(value):
            labels_in_row = str(value).split(',')
            row_list = []
            for lbl in labels_in_row:
                clean_lbl = lbl.strip()
                row_list.append(clean_lbl)
                all_labels_in_category.append(clean_lbl)
            all_rows_original.append(row_list)

    #Count labels to establish baselines
    label_counts_series = pd.Series(all_labels_in_category).value_counts()
    total_occurrences_original = len(all_labels_in_category)
    total_unique_original = len(label_counts_series)

    # We want to test 0 and the chosen cutoff
    cutoffs_to_test = [0, cutoff]

    if cutoff == 0:
        cutoffs_to_test = [0]

    results_for_this_category = []

    #Test both cutoffs
    for c in cutoffs_to_test:
        if c == 0:
            step_name = "0"
        else:
            step_name = str(c)

        # Find valid labels
        valid_labels = []
        for label_name, count in label_counts_series.items():
            if count >= c:
                valid_labels.append(label_name)

        labels_kept_count = len(valid_labels)

        # Clean rows
        filtered_rows = []
        for row in all_rows_original:
            kept_in_row = []
            for label in row:
                if label in valid_labels:
                    kept_in_row.append(label)
            if len(kept_in_row) > 0:
                filtered_rows.append(kept_in_row)

        #Calculate metrics
        if len(filtered_rows) > 0:
            mlb = MultiLabelBinarizer()
            y_matrix = mlb.fit_transform(filtered_rows)

            num_rows = y_matrix.shape[0]
            num_labels = y_matrix.shape[1]
            row_sums = np.sum(y_matrix, axis=1)

            cardinality = np.mean(row_sums)

            total_ones = np.sum(y_matrix)
            density = total_ones / (num_rows * num_labels)
            sparsity = (1 - density) * 100

            label_column_sums = np.sum(y_matrix, axis=0)
            unique_combos = np.unique(y_matrix, axis=0).shape[0]

            # MeanIR
            max_count = np.max(label_column_sums)
            ir_per_label = max_count / label_column_sums
            mean_ir = np.mean(ir_per_label)

            # % kept
            rows_kept_pct = (num_rows / len(all_rows_original)) * 100
            occurrences_kept_pct = (np.sum(label_column_sums) / total_occurrences_original) * 100

            metrics_dict = {
                "Cutoff": step_name,
                "Labels kept": str(labels_kept_count) + " / " + str(total_unique_original),
                "Occurrences kept (%)": round(occurrences_kept_pct, 2),
                "Rows kept (%)": round(rows_kept_pct, 2),
                "Cardinality": round(cardinality, 2),
                "Density": round(density, 4),
                "Sparsity (%)": round(sparsity, 1),
                "Label sets": unique_combos,
                "MeanIR": round(mean_ir, 2)
            }
            results_for_this_category.append(metrics_dict)

    #Make and display table
    df_results = pd.DataFrame(results_for_this_category)
    display(HTML(df_results.to_html(index=False)))


Category: Genre


Cutoff,Labels kept,Occurrences kept (%),Rows kept (%),Cardinality,Density,Sparsity (%),Label sets,MeanIR
0,22 / 22,100.00,100.00,1.80,0.0817,91.8,144,81.24
70,10 / 22,90.41,99.02,1.64,0.1641,83.6,48,3.15


Category: Form


Cutoff,Labels kept,Occurrences kept (%),Rows kept (%),Cardinality,Density,Sparsity (%),Label sets,MeanIR
0,25 / 25,100.00,100.0,1.59,0.0638,93.6,144,125.55
70,10 / 25,90.86,98.0,1.48,0.1479,85.2,63,5.00


Category: Target audience


Cutoff,Labels kept,Occurrences kept (%),Rows kept (%),Cardinality,Density,Sparsity (%),Label sets,MeanIR
0,6 / 6,100.0,100.0,1.3,0.2174,78.3,20,9.3


Category: Topic


Cutoff,Labels kept,Occurrences kept (%),Rows kept (%),Cardinality,Density,Sparsity (%),Label sets,MeanIR
0,218 / 218,100.00,100.00,5.26,0.0241,97.6,2000,44.52
30,97 / 218,85.25,99.54,4.51,0.0465,95.4,1894,7.23


Category: Mood


Cutoff,Labels kept,Occurrences kept (%),Rows kept (%),Cardinality,Density,Sparsity (%),Label sets,MeanIR
0,39 / 39,100.00,100.00,3.62,0.0929,90.7,1180,13.42
80,25 / 39,93.19,99.86,3.38,0.1353,86.5,981,3.23


# 6.0 Create Final Cleaned Dataset

In [ ]:
# Creates the final cleaned dataset by applying the custom cutoffs
#Removes labels that fall below the threshold and drops rows that become completely empty.
#Define individual cutoffs based on EDA
custom_cutoffs = {
    "Genre": 70,
    "Form": 70,
    "Målgruppe": 0,
    "Emne": 30,
    "Stemning": 80
}

labelcols = ["Genre", "Form", "Målgruppe", "Emne", "Stemning"]
df_clean = df.copy()
translations = {"Målgruppe": "Target audience", "Emne": "Topic", "Stemning": "Mood"}

for col in labelcols:
    k = custom_cutoffs[col]
    display_col = translations.get(col, col.title())
    # Count frequencies of each label
    counts = df_clean[col].dropna().astype(str).str.split(",").explode().str.strip().value_counts()
    # Identify labels that meet the threshold
    valid_labels = []
    for label_name, count in counts.items():
        if count >= k:
            valid_labels.append(label_name)

    for i in range(len(df_clean)):
        cell_value = df_clean.at[i, col]

        if pd.notna(cell_value):
            labels = str(cell_value).split(",")
            kept_labels = []

            for lbl in labels:
                clean_lbl = lbl.strip()
                if clean_lbl in valid_labels:
                    kept_labels.append(clean_lbl)

            if len(kept_labels) > 0:
                df_clean.at[i, col] = ", ".join(kept_labels)
            else:
                df_clean.at[i, col] = np.nan

    print(f"{display_col}:")
    print(f"  - {len(valid_labels)}/{len(counts)} labels kept\n")

#Remove rows that have become empty across all 5 categories
rows_before = len(df_clean)
df_clean = df_clean.dropna(subset=labelcols, how="all").reset_index(drop=True)
rows_after = len(df_clean)
print(f"Rows without any labels = {rows_before - rows_after}")
print(f"Final row count after cut = {rows_after}/{len(df)}")

#Save the final dataset
out_name = "tv2_data_cutoffs.csv"
df_clean.to_csv(out_name, index=False)
print(f"\nNew dataset = '{out_name}'")

Genre:
  - 10/22 labels kept

Form:
  - 10/25 labels kept

Target audience:
  - 6/6 labels kept

Topic:
  - 97/218 labels kept

Mood:
  - 25/39 labels kept

Rows without any labels = 0
Final row count after cut = 2153/2153

New dataset = 'tv2_data_cutoffs.csv'
